# Baselines for order selection 
Three baselines run on the IDENTICAL 640 datasets of `ordersweep_validation`
(same crc32 seed formula; every row is hard-paired against the frozen
ordersweep artifact read from Drive). Comparison protocol -- each baseline is
converted to an order decision by a fixed, pre-registered rule:

1. **NID** (neural interaction detection from network weights, Tsang et al.
   2018): a numpy MLP (3-64-32-1, full-batch Adam) is trained per dataset;
   subset strength omega(I) = sum over first-layer units of
   min_j |W1[unit, j in I]| x path strength mu(unit) (product-of-norms
   aggregation through later layers). Decision: k_hat = highest order among
   subsets {12,13,23,123} with omega above tau, where tau is calibrated as
   the 95th percentile of all subset strengths on the o1 cells (NID's own
   null; h standardized per dataset so strengths are comparable). tau is
   stored in metadata.
2. **Pairwise-capped hierarchical baseline** ("pairwise"): hierNet
   (Bien-Tibshirani hierarchical pairwise lasso) via rpy2/R when the R path
   works; otherwise the flagged fallback, an all-pairs lasso on standardized
   [mains, pairwise products] at the 1SE penalty (the CV-optimal penalty
   routinely retains tiny noise coefficients). The fallback is NOT hierNet
   and is never reported as hierNet: metadata and check.txt record which
   implementation ran, and the paper names the one that did. Decision:
   k_hat = 2 iff any interaction coefficient is nonzero at the selected
   penalty, else 1. Both implementations are STRUCTURALLY CAPPED at order
   2: on o3/o3mix cells this baseline cannot be correct, and those rows are
   reported as an expressiveness ceiling, not scored failures -- the paper
   must present it as a pairwise baseline. PRE-REGISTERED prediction: on
   pure o3 the selection is 1, not 2 --
   every pairwise product is population-orthogonal to x1 x2 x3 under all
   dependence conditions in the suite (E[x_i x_j * x1 x2 x3] = 0), so the
   lasso sees no pairwise signal at all.
3. **CV-1SE order selection** on the identical nested poly classes
   (classical rule in the component-selection tradition; this is the
   rule-isolating baseline -- classes fixed, selection rule varies):
   k_hat = smallest k whose 5-fold CV error is within one SE of the best.

Acceptance checks are INTEGRITY checks (performance comparisons are
findings, not checks): (1) every (dgp, dependence, sigma, rep) pairs
exactly with a frozen ordersweep row; (2) NID overselection on o1 is <= 0.15 (the threshold is the 95th
percentile of POOLED null subset strengths, which controls strength-level
exceedances approximately, not an exact 5% row-level rate);
(3) which pairwise implementation ran (rpy2-hierNet or fallback) is
recorded in metadata and check.txt; (4) a config fingerprint is written
next to the artifact and asserted on resume, so a stale per_seed.csv from
an older configuration stops the run instead of being silently extended.
The sweep writes incrementally and resumes if interrupted.

Outputs to `MyDrive/ORDER_SWEEP/results/baselines_validation/`.
Requires the frozen `results/ordersweep_validation/per_seed.csv` on Drive.


In [ ]:
# Cell 1 -- Mount Drive, locate the frozen ordersweep artifact
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/ORDER_SWEEP'
OUT = os.path.join(BASE, 'results', 'baselines_validation')
os.makedirs(OUT, exist_ok=True)
FROZEN = os.path.join(BASE, 'results', 'ordersweep_validation', 'per_seed.csv')
assert os.path.exists(FROZEN), f"frozen ordersweep artifact not found: {FROZEN}"
print('output folder:', OUT)


In [ ]:
# Cell 2 -- Shared data machinery (identical generators and seeds)
import numpy as np, json, csv, time, hashlib, zlib

def make_X(n, dependence, rng):
    if dependence == "indep":
        return rng.standard_normal((n, 3))
    if dependence.startswith("pair"):
        rho = float(dependence[4:])
        x1, x2, z = rng.standard_normal((3, n))
        return np.column_stack([x1, x2, rho * x1 + np.sqrt(1 - rho**2) * z])
    if dependence.startswith("equi"):
        rho = float(dependence[4:])
        g = rng.standard_normal(n)
        z = rng.standard_normal((3, n))
        return (np.sqrt(rho) * g + np.sqrt(1 - rho) * z).T
    raise ValueError(dependence)

def make_h(X, dgp, sigma, rng):
    x1, x2, x3 = X[:, 0], X[:, 1], X[:, 2]
    f = {"o1": x1 + np.tanh(x2) - 0.5 * x3,
         "o2": x1 * x2 + np.tanh(x3),
         "o3": x1 * x2 * x3,
         "o3mix": x1 * x2 * x3 + x1 + x2 * x3}[dgp]
    return f + sigma * rng.standard_normal(len(f))

def cell_rng(dgp, dep, sigma, rep):
    return np.random.default_rng(10_000 + zlib.crc32(f"{dgp}|{dep}|{sigma}".encode()) % 1000 + rep * 977)

TRUE_ORDER = {"o1": 1, "o2": 2, "o3": 3, "o3mix": 3}
N = 20_000
R_REPS = 20
DGPS = ["o1", "o2", "o3", "o3mix"]
DEPS = ["indep", "pair0.5", "pair0.9", "equi0.5"]
SIGMAS = [0.0, 0.5]
SEED_SCHEME = "crc32-v2"
print("shared machinery ready; seed scheme:", SEED_SCHEME)


In [ ]:
# Cell 3 -- Baseline 1: NID on a numpy MLP
def mlp_train(X, h, hidden=(64, 32), epochs=300, lr=1e-2, seed=0, l1_w1=1e-4):
    """Full-batch Adam on a small MLP; returns the weight matrices.
    A light L1 on the first layer follows the sparsity NID assumes."""
    rng = np.random.default_rng(seed)
    n, d = X.shape
    sizes = [d] + list(hidden) + [1]
    W = [rng.standard_normal((a, b)) * np.sqrt(2.0 / a) for a, b in zip(sizes, sizes[1:])]
    b = [np.zeros(s) for s in sizes[1:]]
    mW = [np.zeros_like(w) for w in W]; vW = [np.zeros_like(w) for w in W]
    mB = [np.zeros_like(x) for x in b]; vB = [np.zeros_like(x) for x in b]
    b1, b2, eps = 0.9, 0.999, 1e-8
    hs = (h - h.mean()) / (h.std() + 1e-12)
    for t in range(1, epochs + 1):
        acts = [X]
        for li, (w, bb) in enumerate(zip(W, b)):
            z = acts[-1] @ w + bb
            acts.append(np.maximum(z, 0.0) if li < len(W) - 1 else z)
        pred = acts[-1][:, 0]
        gout = (2.0 / n) * (pred - hs)
        grad = gout[:, None]
        gW, gB = [None] * len(W), [None] * len(b)
        for li in range(len(W) - 1, -1, -1):
            gW[li] = acts[li].T @ grad
            gB[li] = grad.sum(0)
            if li > 0:
                grad = (grad @ W[li].T) * (acts[li] > 0)
        gW[0] = gW[0] + l1_w1 * np.sign(W[0])
        for li in range(len(W)):
            for g, w, m, v in [(gW[li], W[li], mW[li], vW[li]),
                               (gB[li], b[li], mB[li], vB[li])]:
                m *= b1; m += (1 - b1) * g
                v *= b2; v += (1 - b2) * g * g
                w -= lr * (m / (1 - b1**t)) / (np.sqrt(v / (1 - b2**t)) + eps)
    return W

SUBSETS = [(0, 1), (0, 2), (1, 2), (0, 1, 2)]

def nid_strengths(W):
    """omega(I) = sum_units min_{j in I} |W1[j, unit]| * mu(unit), where
    mu aggregates outgoing influence by products of column norms."""
    W1 = np.abs(W[0])                      # (d, H1)
    mu = np.abs(W[1])                      # (H1, H2)
    for w in W[2:]:
        mu = mu @ np.abs(w)
    mu = mu[:, 0] if mu.ndim == 2 else mu  # (H1,)
    out = {}
    for I in SUBSETS:
        out[I] = float(np.sum(np.min(W1[list(I), :], axis=0) * mu))
    return out

def nid_khat(strengths, tau):
    orders = [len(I) for I, s in strengths.items() if s > tau]
    return max(orders) if orders else 1
print("NID module ready")


In [ ]:
# Cell 4 -- Baseline 2: pairwise-capped baseline (hierNet via rpy2, or flagged all-pairs 1SE lasso)
PAIRWISE_IMPL = None
try:
    import rpy2.robjects as ro
    from rpy2.robjects import numpy2ri
    numpy2ri.activate()
    try:
        ro.r("suppressMessages(library(hierNet))")
    except Exception:
        print("installing hierNet in R (a few minutes, once per runtime)...")
        ro.r("install.packages('hierNet', repos='https://cloud.r-project.org', quiet=TRUE)")
        ro.r("suppressMessages(library(hierNet))")
    PAIRWISE_IMPL = "rpy2-hierNet"
    def pairwise_khat(X, h, seed=0):
        sub = np.random.default_rng(seed).permutation(len(X))[:4000]  # hierNet scales poorly in n
        ro.globalenv["x"] = X[sub]; ro.globalenv["y"] = h[sub]
        # numpy2ri delivers R arrays; hierNet's class checks require plain
        # matrix/numeric, so coerce inside R before fitting
        ro.r("x <- as.matrix(x); y <- as.numeric(y)")
        ro.r("fit <- hierNet.path(x, y, trace=0)")
        ro.r("cv <- hierNet.cv(fit, x, y, nfolds=5, trace=0)")
        nnz = ro.r("f2 <- hierNet(x, y, lam=cv$lamhat); sum(abs(f2$th) > 1e-8)")[0]
        return 2 if nnz > 0 else 1
except Exception as e:
    print("rpy2/hierNet unavailable -> flagged fallback (all-pairs LassoCV):", type(e).__name__)
    PAIRWISE_IMPL = "allpairs-lasso-1se (NOT hierNet)"
    from sklearn.linear_model import LassoCV, Lasso
    def pairwise_khat(X, h, seed=0):
        Z = np.column_stack([X, X[:, 0]*X[:, 1], X[:, 0]*X[:, 2], X[:, 1]*X[:, 2]])
        Z = (Z - Z.mean(0)) / np.where(Z.std(0) == 0, 1.0, Z.std(0))
        hs = (h - h.mean()) / (h.std() + 1e-12)
        cv = LassoCV(cv=5, alphas=40, random_state=seed).fit(Z, hs)
        # 1SE rule: largest penalty whose CV error is within one SE of the best
        mse = cv.mse_path_.mean(axis=1)
        se = cv.mse_path_.std(axis=1, ddof=1) / np.sqrt(cv.mse_path_.shape[1])
        best = int(np.argmin(mse))
        ok = mse <= mse[best] + se[best]
        alpha_1se = float(cv.alphas_[np.argmax(ok)])   # alphas_ is descending
        fit = Lasso(alpha=alpha_1se).fit(Z, hs)
        return 2 if np.any(np.abs(fit.coef_[3:]) > 1e-8) else 1
print("pairwise implementation:", PAIRWISE_IMPL)


In [ ]:
# Cell 5 -- Baseline 3: CV-1SE order selection on the identical poly classes
from itertools import product as iproduct

def monomial_exps(d, D, max_active):
    out = []
    for combo in iproduct(range(D + 1), repeat=d):
        if sum(combo) <= D and sum(1 for c in combo if c > 0) <= max_active:
            out.append(combo)
    return out

def poly_design(X, D, max_active):
    d = X.shape[1]
    exps = monomial_exps(d, D, max_active)
    cols = []
    for e in exps:
        col = np.ones(X.shape[0])
        for j, p in enumerate(e):
            if p > 0:
                col = col * X[:, j] ** p
        cols.append(col)
    return np.column_stack(cols), exps.index(tuple([0] * d))

def cv1se_khat(X, h, K=3, D=4, folds=5, seed=0):
    n = X.shape[0]
    idx = np.random.default_rng(seed).permutation(n)
    fold_id = np.arange(n) % folds
    fold_id = fold_id[np.argsort(idx)]      # deterministic shuffled folds
    errs = {k: [] for k in range(1, K + 1)}
    for k in range(1, K + 1):
        Phi, ci = poly_design(X, D, k)
        for f in range(folds):
            tr, te = fold_id != f, fold_id == f
            mu = Phi[tr].mean(0); sd = Phi[tr].std(0); sd[sd == 0] = 1.0
            P = (Phi - mu) / sd; P[:, ci] = 1.0
            hm = h[tr].mean()
            beta, *_ = np.linalg.lstsq(P[tr], h[tr] - hm, rcond=None)
            resid = (h[te] - hm) - P[te] @ beta
            errs[k].append(float(np.mean(resid ** 2)))
    means = {k: np.mean(v) for k, v in errs.items()}
    ses = {k: np.std(v, ddof=1) / np.sqrt(folds) for k, v in errs.items()}
    kbest = min(means, key=means.get)
    thresh = means[kbest] + ses[kbest]
    return min(k for k in range(1, K + 1) if means[k] <= thresh)
print("CV-1SE module ready")


In [ ]:
# Cell 6 -- NID calibration on o1 (its own null), then the paired sweep with resume
EXP = "baselines_validation"
PS = os.path.join(OUT, "per_seed.csv")

# frozen ordersweep khat for pairing
frozen = {}
with open(FROZEN) as f:
    for r in csv.DictReader(f):
        frozen[(r["dgp"], r["dependence"], r["sigma"], r["rep"])] = int(r["khat"])
assert len(frozen) == 640, f"frozen artifact has {len(frozen)} rows"

# calibration pass: all o1 replicates (tau is deterministic given seeds,
# so it is cached to Drive and reloaded after a runtime restart)
TAU_PATH = os.path.join(OUT, "nid_tau.json")
if os.path.exists(TAU_PATH):
    TAU = json.load(open(TAU_PATH))["tau"]
    print(f"NID tau loaded from cache: {TAU:.5f}")
else:
    t0 = time.time()
    cal = []
    for dep in DEPS:
        for sigma in SIGMAS:
            for rep in range(R_REPS):
                rng = cell_rng("o1", dep, sigma, rep)
                X = make_X(N, dep, rng); h = make_h(X, "o1", sigma, rng)
                W = mlp_train(X, h, seed=rep)
                cal.extend(nid_strengths(W).values())
    TAU = float(np.quantile(cal, 0.95))
    with open(TAU_PATH, "w") as f:
        json.dump({"tau": TAU, "n_strengths": len(cal)}, f)
    print(f"NID tau (95th pct of {len(cal)} null strengths): {TAU:.5f}  ({time.time()-t0:.0f}s)")

# config fingerprint: refuse to resume onto an artifact from another config
CONFIG_FP = hashlib.sha256(json.dumps(
    {"N": N, "R_REPS": R_REPS, "DGPS": DGPS, "DEPS": DEPS, "SIGMAS": SIGMAS,
     "SEED_SCHEME": SEED_SCHEME, "TAU": TAU, "PAIRWISE_IMPL": PAIRWISE_IMPL},
    sort_keys=True).encode()).hexdigest()
FP_PATH = os.path.join(OUT, "config_fingerprint.json")
if os.path.exists(PS):
    assert os.path.exists(FP_PATH), \
        "per_seed.csv exists without a fingerprint (older run): delete or rename the output folder"
    stored = json.load(open(FP_PATH))["fingerprint"]
    assert stored == CONFIG_FP, \
        "per_seed.csv is from a DIFFERENT configuration: delete or rename the output folder"
else:
    with open(FP_PATH, "w") as f:
        json.dump({"fingerprint": CONFIG_FP}, f)

done = set()
if os.path.exists(PS):
    with open(PS) as f:
        for r in csv.DictReader(f):
            done.add((r["dgp"], r["dependence"], r["sigma"], r["rep"]))
    print(f"resuming: {len(done)} rows already computed")

fields = ["experiment","dgp","dependence","sigma","rep","true_order",
          "khat_ordersweep","khat_nid","khat_pairwise","khat_cv1se"]
mode = "a" if done else "w"
fout = open(PS, mode, newline="")
w = csv.DictWriter(fout, fieldnames=fields)
if not done: w.writeheader()
t0 = time.time()
for dgp in DGPS:
    for dep in DEPS:
        for sigma in SIGMAS:
            for rep in range(R_REPS):
                key = (dgp, dep, str(sigma), str(rep))
                if key in done: continue
                rng = cell_rng(dgp, dep, sigma, rep)
                X = make_X(N, dep, rng); h = make_h(X, dgp, sigma, rng)
                W = mlp_train(X, h, seed=rep)
                k_nid = nid_khat(nid_strengths(W), TAU)
                k_pw = pairwise_khat(X, h, seed=rep)
                k_cv = cv1se_khat(X, h, seed=rep)
                w.writerow({"experiment": EXP, "dgp": dgp, "dependence": dep,
                            "sigma": sigma, "rep": rep, "true_order": TRUE_ORDER[dgp],
                            "khat_ordersweep": frozen[(dgp, dep, str(float(sigma)), str(rep))]
                                if (dgp, dep, str(float(sigma)), str(rep)) in frozen
                                else frozen[(dgp, dep, str(sigma), str(rep))],
                            "khat_nid": k_nid, "khat_pairwise": k_pw, "khat_cv1se": k_cv})
                fout.flush()
            print(f"{dgp:6s} {dep:8s} sigma={sigma:3.1f} done ({time.time()-t0:6.0f}s)", flush=True)
fout.close()

with open(os.path.join(OUT, "metadata.json"), "w") as f:
    json.dump({"experiment": EXP, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
               "config": {"N": N, "R_REPS": R_REPS, "DGPS": DGPS, "DEPS": DEPS,
                          "SIGMAS": SIGMAS, "SEED_SCHEME": SEED_SCHEME,
                          "nid": {"hidden": [64, 32], "epochs": 300, "lr": 1e-2,
                                  "l1_w1": 1e-4, "tau": TAU, "calibration": "95th pct on o1"},
                          "pairwise_impl": PAIRWISE_IMPL,
                          "cv1se": {"folds": 5, "D": 4, "K": 3}},
               "code_sha256": CODE_SHA_B,
               "provenance_scheme": "source-v2",
               "provenance_scope": "full function source; config carries all constants incl. tau and seed scheme",
               "numpy": np.__version__}, f, indent=2)
print("wrote per_seed.csv, metadata.json")


In [ ]:
# Cell 7 -- Verification, integrity checks, comparison report
import csv as _csv
rows = list(_csv.DictReader(open(PS)))
assert all(r["experiment"] == "baselines_validation" for r in rows), "stamp mismatch"

checks, story = [], []
checks.append((f"pairing: {len(rows)} rows, one per frozen ordersweep replicate (expect 640)",
               len(rows) == 640))
lo_nid = [r for r in rows if r["dgp"] == "o1"]
nid_over = sum(int(r["khat_nid"]) > 1 for r in lo_nid) / len(lo_nid)
checks.append((f"NID calibration on o1: overselection {nid_over:.3f} <= 0.15 "
               "(threshold = 95th pct of pooled null subset strengths; row-level rate approximate)",
               nid_over <= 0.15))
import json as _json
_impl = _json.load(open(os.path.join(OUT, "metadata.json")))["config"]["pairwise_impl"] \
        if os.path.exists(os.path.join(OUT, "metadata.json")) else PAIRWISE_IMPL
checks.append((f"pairwise implementation recorded: {_impl}", True))
for name, ok in checks:
    line = ("PASS  " if ok else "FAIL  ") + name
    story.append(line); print(line)

METHODS = ["ordersweep", "nid", "pairwise", "cv1se"]
print(f"\n{'dgp':7s}{'dep':9s}{'sig':>4s}" + "".join(f"{m:>12s}" for m in METHODS) + "   (correct-rate)")
for dgp in DGPS:
    for dep in DEPS:
        for sigma in SIGMAS:
            cs = [r for r in rows if r["dgp"]==dgp and r["dependence"]==dep and float(r["sigma"])==sigma]
            t = int(cs[0]["true_order"])
            line = f"{dgp:7s}{dep:9s}{sigma:4.1f}"
            for m in METHODS:
                rate = sum(int(r[f'khat_{m}'])==t for r in cs)/len(cs)
                cell_s = f"{rate:.2f}*" if (m == "pairwise" and t == 3) else f"{rate:.2f}"
                line += f"{cell_s:>12s}"
            print(line)
    print()

for m in METHODS:
    corr = sum(int(r[f"khat_{m}"]) == int(r["true_order"]) for r in rows) / len(rows)
    over = sum(int(r[f"khat_{m}"]) > int(r["true_order"]) for r in rows) / len(rows)
    under = sum(int(r[f"khat_{m}"]) < int(r["true_order"]) for r in rows) / len(rows)
    line = f"OBS   {m}: overall correct {corr:.3f}, over {over:.3f}, under {under:.3f}"
    story.append(line); print(line)
print("      * pairwise baseline is structurally capped at order 2: o3/o3mix rows are an")
print("        expressiveness ceiling, not scored failures")
o3h = [r for r in rows if r["dgp"] == "o3"]
k1 = sum(int(r["khat_pairwise"]) == 1 for r in o3h)
story.append(f"OBS   pairwise baseline on pure o3: khat=1 in {k1}/{len(o3h)} (pre-registered orthogonality prediction; order-2 ceiling is structural)")
print(story[-1])
with open(os.path.join(OUT, "check.txt"), "w") as f:
    f.write("\n".join(story) + "\n")
print("\nwrote check.txt")
